# Step 4 — Sample-Level Pseudobulk Clustering

## The problem this step solves

You have 39 cleaned samples ready to integrate into a single atlas. Integration is expensive — it takes hours of compute for 200k cells. Before you commit to that, you want to answer one question: **do my samples actually contain the biology I think they contain, and do they group the way I expect?**

This step is a sanity check, not a filtering step. Nothing gets removed. The question is: when you reduce each sample to a single average expression profile and compare them, do scWAT samples cluster with scWAT, vWAT with vWAT, SkM with SkM? If they do, your samples are well-behaved and integration will work cleanly. If a scWAT sample clusters with SkM, you have a labeling error, a sample swap, or a severe dissociation failure.

## Why this matters for the Yang et al. paper

The paper confirmed that **the major drivers of variation were tissue, diet, and exercise — not batch** (Figure S1F). Verifying this before integration is what lets you choose `merge` (simple concatenation) rather than aggressive batch correction. If you had skipped this check and integration showed samples clustering by collection day rather than biology, you would not know whether Harmony was fixing a real batch effect or regressing out a biological signal you cared about.

The pseudobulk clustering also revealed that **vWAT was more strongly affected by HFD than scWAT** — even at this coarse, sample-level view, obese visceral fat samples separated from control visceral fat more cleanly than their subcutaneous counterparts. This finding held and deepened at the single-cell level: vWAT showed 7.6× more cell-state-level DEGs than scWAT in the obesity comparison.

## What "pseudobulk" means and why it works

"Pseudobulk" means collapsing a single-cell sample into one expression vector by averaging across all its cells — treating it like a bulk RNA-seq sample. This is useful for sample-level QC because:

1. The mean expression is **robust** — it reflects the overall transcriptional identity of the sample, not any single cell type
2. **Spearman correlation** between pseudobulk profiles measures global transcriptional similarity; it is sensitive to sample swaps and gross failures
3. The structure is **interpretable**: within-tissue correlations should be > 0.95, between-tissue correlations < 0.7

> **ML analogy:** This is like computing dataset-level statistics (mean feature vector per dataset) and checking whether datasets from the same class are similar. Before merging datasets into a training pool, you verify they are from compatible distributions.

## What this notebook does

1. Load each per-sample `.h5ad` file from Step 3
2. Compute per-gene mean expression across all cells in each sample (pseudobulk)
3. Build a genes × samples matrix
4. Compute Spearman correlation between all sample pairs
5. Generate a hierarchical clustering heatmap with phenotype annotations
6. Report numeric within- vs. between-tissue correlation summary

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import scipy.cluster.hierarchy as sch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path

## Configuration

- `input_file_directory` — directory containing the per-sample `.h5ad` files produced by Step 3.
- `pheno_df` — a DataFrame with sample names as the index and one or more phenotype columns (e.g., `tissue`, `diet`, `exercise`). These become the color-coded annotation bars on the heatmap.
- `pheno_colors` — a dict mapping each phenotype column to a dict of `{category: color}`. If `None`, colors are assigned automatically.
- `file_name` — prefix for the output PDF.

In [ ]:
input_file_directory = Path(".")
target_folder        = Path(".")
file_name            = "all_samples"

# Example phenotype table — replace with your actual metadata.
# Row index must match sample names (the .h5ad filename stem, e.g. "lib_001").
pheno_df = pd.DataFrame({
    "tissue":    {"lib_001": "scWAT", "lib_002": "scWAT", "lib_003": "vWAT", "lib_004": "SkM"},
    "diet":      {"lib_001": "HFD",   "lib_002": "NCD",   "lib_003": "HFD",  "lib_004": "NCD"},
    "exercise":  {"lib_001": "Yes",   "lib_002": "No",    "lib_003": "No",   "lib_004": "Yes"},
})

# Optional: specify colors for each phenotype category.
# If None, seaborn will assign colors automatically.
pheno_colors = {
    "tissue":   {"scWAT": "#4DAF4A", "vWAT": "#377EB8", "SkM": "#E41A1C"},
    "diet":     {"HFD": "#FF7F00",   "NCD": "#984EA3"},
    "exercise": {"Yes": "#A65628",   "No":  "#F781BF"},
}

## Step 1 — Build the pseudobulk expression matrix

### Why mean expression, not sum?

The R script uses `rowMeans` on the SCT (SCTransform-normalized) assay. Mean expression is preferable to sum here because:
- **Sum scales with cell count** — a sample with 5,000 cells will have roughly 5× the summed counts of a sample with 1,000 cells, even if they are biologically identical. This would dominate the correlation structure.
- **Mean is depth-normalized** — it reflects the average transcriptional state of the sample regardless of how many cells were profiled.

We use the normalized `.X` layer from the Step 3 output (which stored raw counts in `layers["counts"]` and kept normalized values in `.X`).

In [ ]:
h5ad_files = sorted(input_file_directory.glob("*_processed.h5ad"))
if not h5ad_files:
    raise FileNotFoundError(
        f"No *_processed.h5ad files found in {input_file_directory}. "
        "Check that Step 3 has been run and the path is correct."
    )
print(f"Found {len(h5ad_files)} sample files")

pseudobulk_columns = {}

for fpath in h5ad_files:
    sample_name = fpath.stem.replace("_processed", "")
    print(f"  Loading: {sample_name}")

    adata = sc.read_h5ad(fpath)

    # Compute mean expression across all cells for each gene.
    # scipy sparse matrices require .toarray() or use .mean(axis=0) directly.
    if sp.issparse(adata.X):
        mean_expr = np.array(adata.X.mean(axis=0)).flatten()
    else:
        mean_expr = adata.X.mean(axis=0)

    pseudobulk_columns[sample_name] = pd.Series(mean_expr, index=adata.var_names)

# Build genes × samples DataFrame.
# All samples must share the same gene set (guaranteed if they were processed
# from the same CellRanger reference). inner join handles any minor discrepancies.
bulk_df = pd.DataFrame(pseudobulk_columns)
bulk_df = bulk_df.dropna()   # drop genes missing in any sample

print(f"\nPseudobulk matrix: {bulk_df.shape[0]:,} genes × {bulk_df.shape[1]} samples")
bulk_df.head()

## Step 2 — Compute Spearman correlation between samples

### Why Spearman, not Pearson?

Spearman correlation measures the **rank-order** relationship between two expression profiles. It is used here instead of Pearson because:

- **Robustness to outlier genes** — a few very highly expressed genes (e.g., hemoglobin, collagen) can dominate a Pearson correlation and make biologically different samples look similar just because they share the same dominant transcript. Spearman down-weights these by converting values to ranks.
- **Non-linearity tolerance** — the relationship between expression values across samples is not strictly linear after normalization; Spearman captures monotonic relationships more faithfully.
- **Standard practice** — Spearman is the recommended metric for bulk and pseudobulk sample-level QC in the Bioconductor community (e.g., `DESeq2` and `edgeR` vignettes use it for this purpose).

The resulting matrix is samples × samples, with values between −1 and 1. For high-quality scRNA-seq data from the same tissue, within-tissue correlations are typically > 0.95.

In [ ]:
# pandas .corr(method='spearman') computes pairwise Spearman correlation
# across columns (samples), using genes as observations.
corr_matrix = bulk_df.corr(method="spearman")

print("Correlation matrix (samples × samples):")
print(corr_matrix.round(3))

## Step 3 — Hierarchical clustering heatmap

### Distance metric and linkage method

The R script uses `1 - correlation` as the distance metric and **Ward's linkage** (`ward.D`) for hierarchical clustering.

- **Distance = 1 − ρ** — converts Spearman correlation (similarity) to a distance (dissimilarity). Two perfectly correlated samples have distance 0; uncorrelated samples have distance 1.
- **Ward's linkage** — at each merge step, joins the two clusters whose union minimizes the total within-cluster variance. Ward's tends to produce compact, roughly equal-sized clusters and is well-suited to datasets where clusters have similar sizes. It is more conservative than average or complete linkage and less susceptible to chaining artifacts.

### Reading the heatmap

- **Color intensity** — darker blue = higher Spearman correlation = more similar global transcriptomes.
- **Dendrogram** — the tree structure shows which samples are most similar. Samples should cluster by tissue first (the largest transcriptional differences), then by condition within tissue.
- **Annotation bars** — the colored bars along the top/left show phenotype labels. If clustering aligns with annotation colors, the experiment has the expected signal.
- **Red flags:**
  - A sample from one tissue clustering with a different tissue
  - One sample visibly lighter (less correlated) than all others in its group — may indicate a failed library or sample swap
  - Clustering driven entirely by a non-biological variable (e.g., sequencing batch) rather than tissue or condition

In [ ]:
# Build distance matrix from correlation: dist = 1 - rho
dist_matrix = 1 - corr_matrix.values
np.fill_diagonal(dist_matrix, 0)   # ensure exact zeros on diagonal

# Hierarchical clustering with Ward linkage — matches R's ward.D
linkage = sch.linkage(sch.distance.squareform(dist_matrix), method="ward")
sample_order = sch.leaves_list(linkage)   # reordered sample indices

# Build annotation DataFrame aligned to the correlation matrix columns
# Only include phenotypes whose index overlaps with our sample names
shared_samples = [s for s in corr_matrix.columns if s in pheno_df.index]
annot_df = pheno_df.loc[shared_samples] if shared_samples else None

# Build color lookup for annotations
# Each phenotype column gets a palette; we create a row_colors DataFrame
# that seaborn.clustermap uses as side/top color bars.
def make_color_lut(series, color_dict=None):
    """Map a categorical series to colors, using color_dict if provided."""
    categories = series.unique()
    if color_dict:
        return series.map(color_dict)
    palette = sns.color_palette("tab10", len(categories))
    lut = dict(zip(categories, palette))
    return series.map(lut)

if annot_df is not None:
    col_colors = pd.DataFrame({
        col: make_color_lut(annot_df[col], pheno_colors.get(col) if pheno_colors else None)
        for col in annot_df.columns
    })
else:
    col_colors = None

# Determine figure size proportional to number of samples,
# matching the R formula: width = 0.2 * n, height = 0.15 * n (scaled up)
n = corr_matrix.shape[0]
fig_w = max(6, 0.6 * n)
fig_h = max(5, 0.5 * n)

g = sns.clustermap(
    corr_matrix,
    method="ward",
    metric="euclidean",          # applied to the correlation values directly;
                                  # we pass precomputed linkage below
    row_linkage=linkage,
    col_linkage=linkage,
    col_colors=col_colors,
    cmap=sns.color_palette("Blues", as_cmap=True),
    vmin=corr_matrix.values[~np.eye(n, dtype=bool)].min(),  # exclude diagonal
    vmax=1.0,
    linewidths=0,
    xticklabels=True,
    yticklabels=False,
    figsize=(fig_w, fig_h),
)

g.ax_heatmap.set_title("Sample-level pseudobulk Spearman correlation", pad=12)

# Add legend patches for each phenotype
if pheno_colors and annot_df is not None:
    legend_handles = []
    for pheno_col, color_dict in pheno_colors.items():
        for label, color in color_dict.items():
            legend_handles.append(mpatches.Patch(color=color, label=f"{pheno_col}: {label}"))
    g.ax_heatmap.legend(
        handles=legend_handles, bbox_to_anchor=(1.25, 1),
        loc="upper left", borderaxespad=0, fontsize=8
    )

out_path = target_folder / f"{file_name}_sample_level_pseudo_bulk_clustering.pdf"
g.savefig(out_path, bbox_inches="tight")
print(f"Saved: {out_path}")
plt.show()

## Step 4 — Numeric summary of within- and between-group correlations

The heatmap gives a visual overview, but it can be useful to also see the numeric correlation distributions split by phenotype — especially when the number of samples is large and the heatmap is dense.

A healthy dataset should show clearly higher within-tissue correlations than between-tissue correlations.

In [ ]:
if annot_df is not None and "tissue" in annot_df.columns:
    samples = corr_matrix.columns.tolist()
    records = []
    for i, s1 in enumerate(samples):
        for j, s2 in enumerate(samples):
            if j <= i:   # upper triangle only
                continue
            t1 = annot_df.loc[s1, "tissue"] if s1 in annot_df.index else "unknown"
            t2 = annot_df.loc[s2, "tissue"] if s2 in annot_df.index else "unknown"
            records.append({
                "sample_1":     s1,
                "sample_2":     s2,
                "tissue_1":     t1,
                "tissue_2":     t2,
                "comparison":   "within-tissue" if t1 == t2 else "between-tissue",
                "spearman_rho": corr_matrix.loc[s1, s2],
            })

    corr_summary = pd.DataFrame(records)
    print(corr_summary.groupby("comparison")["spearman_rho"].describe().round(3))
    print()

    # Distribution plot
    fig, ax = plt.subplots(figsize=(6, 4))
    for comparison, group in corr_summary.groupby("comparison"):
        ax.hist(group["spearman_rho"], bins=15, alpha=0.6, label=comparison)
    ax.set_xlabel("Spearman ρ")
    ax.set_ylabel("Count")
    ax.set_title("Within- vs. between-tissue pseudobulk correlations")
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No 'tissue' column in pheno_df — skipping within/between group summary.")

## Interpreting the results and deciding whether to proceed

### Green light conditions
- All samples cluster primarily by tissue
- Within-tissue Spearman ρ is consistently > 0.90
- No sample is visibly isolated from its group

### Investigate further if
- A sample clusters with the wrong tissue → check sample labels in the manifest and CellRanger alignment stats from Step 1
- One sample within a tissue has unusually low correlations (< 0.85) with its peers → check its Step 3 QC metrics; it may have failed the library prep or had poor dissociation
- All samples cluster by a technical variable (flowcell, date) rather than tissue → the batch effect is large and will need to be addressed in Step 5 integration (use `rPCA` or `CCA` rather than `merge`)

This step does not remove any samples. Its purpose is to give you confidence — or a warning — before committing to the time-intensive multi-sample integration in Step 5.